# Model 7 — Per-Nozzle SD Prediction with Neural Network (improved extrapolation)
*CRISP-DM Phase 1 — Business Understanding*

Model 6 (Random Forest) predicts the SD value for every individual nozzle — but RF **cannot extrapolate**
to voltage or flow-rate levels outside the training range.

This model uses a **Neural Network (MLP)** with three improvements over a naive MLP:

1. **No polynomial features (V², Fr²)** — polynomial features amplify the out-of-distribution shift and caused the naive MLP to predict negative (impossible) values at V=34. The MLP learns non-linearity through its hidden layers and doesn't need V².
2. **tanh activation instead of ReLU** — tanh is smoother and more stable outside the training range than ReLU.
3. **Output clipped to ≥0** — SD values are physically always positive.

RF still uses the full feature set including V² and Fr² (it benefits from them within the training range).

## 1. Load and reshape
*CRISP-DM Phase 3 — Data Preparation*

Same data and reshape logic as Model 6. 5% sample = ~6 million rows.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

df = pd.read_parquet('../../data/new/extended-rows.parquet 2')

value_cols = [c for c in df.columns if 'Value' in c]
df = df.dropna(subset=['Color$'])
df = df.sample(frac=0.05, random_state=42).reset_index(drop=True)

n_cond    = len(df)
n_nozzles = len(value_cols)

print(f'Conditions sampled: {n_cond}')
print(f'Nozzles per condition: {n_nozzles:,}')
print(f'Total rows: {n_cond * n_nozzles:,}')

## 2. Build feature matrix
*CRISP-DM Phase 3 — Data Preparation*

Two feature sets:
- **X** (11 features, for RF): includes V², Fr² — polynomial terms help RF within training range
- **X_mlp** (9 features, for MLP): drops V² and Fr² — MLP learns curvature through layers, polynomial features hurt extrapolation

In [ ]:
vals = df[value_cols].values.astype(np.float32)

chips   = np.array([int(float(c.split('_')[0]))           for c in value_cols], dtype=np.int16)
nozzles = np.array([int(c.split('_')[-1].replace('#','')) for c in value_cols], dtype=np.int16)

# Color$ is stored as float in this parquet (0=M, 1=C, 2=Y, 3=K)
color_map = {'M': 0, 'C': 1, 'Y': 2, 'K': 3}
color_enc = df['Color$'].astype(float).values.astype(np.int8)
cond_arr  = df[['V', 'F_r', 'dt2', 'Coverage#']].values.astype(np.float32)

V_rep    = np.repeat(cond_arr[:, 0], n_nozzles)
Fr_rep   = np.repeat(cond_arr[:, 1], n_nozzles)
dt2_rep  = np.repeat(cond_arr[:, 2], n_nozzles)
cov_rep  = np.repeat(cond_arr[:, 3], n_nozzles)
col_rep  = np.repeat(color_enc,       n_nozzles).astype(np.float32)
chip_rep = np.tile(chips,   n_cond).astype(np.float32)
noz_rep  = np.tile(nozzles, n_cond).astype(np.float32)
y        = vals.flatten()

X = np.column_stack([
    V_rep, Fr_rep, dt2_rep, cov_rep, col_rep,
    V_rep * Fr_rep,
    dt2_rep * cov_rep,
    V_rep ** 2,
    Fr_rep ** 2,
    chip_rep,
    noz_rep
]).astype(np.float32)

MLP_COLS = [0, 1, 2, 3, 4, 5, 6, 9, 10]
X_mlp = X[:, MLP_COLS]

feature_names     = ['V','F_r','dt2','Coverage','color','V_x_Fr','dt2_x_cov','V_sq','Fr_sq','HeadIdx','NozzleIdx']
feature_names_mlp = ['V','F_r','dt2','Coverage','color','V_x_Fr','dt2_x_cov','HeadIdx','NozzleIdx']

print(f'RF  feature matrix: {X.shape}')
print(f'MLP feature matrix: {X_mlp.shape}')
print(f'Color$ unique values in data: {sorted(df["Color$"].unique())}  (0=M, 1=C, 2=Y, 3=K)')
print(f'Color distribution: {dict(zip(*np.unique(color_enc, return_counts=True)))}')

## 3. Train Neural Network + RF baseline
*CRISP-DM Phase 4 — Modeling*

- **RF**: trained on full 11-feature X (with V², Fr²)
- **MLP**: trained on 9-feature X_mlp (no V², Fr²), tanh activation, output clipped to ≥0

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_mlp_train = X_train[:, MLP_COLS]
X_mlp_test  = X_test[:,  MLP_COLS]
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

print('\nTraining Random Forest...')
rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1, max_samples=0.5)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
r2_rf     = r2_score(y_test, y_pred_rf)
mae_rf    = mean_absolute_error(y_test, y_pred_rf)
print(f'RF  →  R²={r2_rf:.4f}  MAE={mae_rf:.6f}')

print('\nTraining Neural Network...')
mlp_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp',    MLPRegressor(
        hidden_layer_sizes=(128, 64),
        activation='tanh',
        batch_size=5000,
        max_iter=200,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=5,
        validation_fraction=0.1,
        learning_rate_init=0.001,
        verbose=False
    ))
])
mlp_pipe.fit(X_mlp_train, y_train)
y_pred_mlp_raw = mlp_pipe.predict(X_mlp_test)
y_pred_mlp     = np.clip(y_pred_mlp_raw, 0, None)
r2_mlp         = r2_score(y_test, y_pred_mlp)
mae_mlp        = mean_absolute_error(y_test, y_pred_mlp)
mlp_model      = mlp_pipe.named_steps['mlp']
print(f'MLP →  R²={r2_mlp:.4f}  MAE={mae_mlp:.6f}  (iterations: {mlp_model.n_iter_})')

print(f'\n{"Model":<20} {"R²":>8} {"MAE":>12} {"Notes":<30}')
print(f'{"Random Forest":<20} {r2_rf:>8.4f} {mae_rf:>12.6f} {"full features, no extrapolation":<30}')
print(f'{"Neural Network":<20} {r2_mlp:>8.4f} {mae_mlp:>12.6f} {"9 features, tanh, clipped ≥0":<30}')

In [ ]:
import joblib, os
os.makedirs('../../models', exist_ok=True)
joblib.dump(mlp_pipe, '../../models/mlp_nozzle_sd.pkl')
print('Saved: models/mlp_nozzle_sd.pkl')

## 4. Neural Network training curve
*CRISP-DM Phase 4 — Modeling*

Shows training loss and validation R² per iteration. Early stopping prevents overfitting.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(mlp_model.loss_curve_, color='#00b4d8', linewidth=2, label='Training loss')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss (MSE)')
ax.set_title(f'Neural Network training — {mlp_model.n_iter_} iterations\n'
             f'early stopping after {mlp_model.n_iter_no_change} iterations without improvement')
ax.grid(alpha=0.3)

if hasattr(mlp_model, 'validation_scores_') and mlp_model.validation_scores_:
    ax2 = ax.twinx()
    ax2.plot(mlp_model.validation_scores_, color='#e74c3c', linewidth=2,
             linestyle='--', label='Validation R²')
    ax2.set_ylabel('Validation R²', color='#e74c3c')
    ax2.tick_params(axis='y', colors='#e74c3c')
    ax2.legend(loc='center right')

ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 5. Predicted vs Actual — RF vs MLP per color
*CRISP-DM Phase 5 — Evaluation*

Two rows: top = Random Forest, bottom = Neural Network. Four columns = four ink colors.

In [ ]:
color_names = {0:'Magenta', 1:'Cyan', 2:'Yellow', 3:'Black'}
dot_colors  = {0:'#e91e8c', 1:'#00b4d8', 2:'#f4c430', 3:'#444444'}
color_test  = X_test[:, 4].astype(int)

models_cmp  = [('Random Forest', y_pred_rf, r2_rf), ('Neural Network', y_pred_mlp, r2_mlp)]

fig, axes = plt.subplots(2, 4, figsize=(18, 10))

for row_idx, (model_name, y_pred, r2_overall) in enumerate(models_cmp):
    for col_idx, c in enumerate([0, 1, 2, 3]):
        ax = axes[row_idx, col_idx]
        mask = color_test == c
        yt, yp = y_test[mask], y_pred[mask]
        if len(yt) == 0:
            ax.set_visible(False); continue
        idx = np.random.choice(len(yt), min(20000, len(yt)), replace=False)
        r2_c = r2_score(yt, yp)
        lim  = [min(yt.min(), yp.min()) - 0.001, max(yt.max(), yp.max()) + 0.001]
        ax.scatter(yt[idx], yp[idx], alpha=0.05, s=2, color=dot_colors[c])
        ax.plot(lim, lim, 'r--', linewidth=1.2)
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_xlabel('Actual SD')
        ax.set_ylabel('Predicted SD')
        ax.set_title(f'{model_name}\n{color_names[c]}  R²={r2_c:.3f}')

plt.suptitle('Predicted vs Actual per color — RF (top) vs Neural Network (bottom)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Extrapolation — predicting untested voltage levels
*CRISP-DM Phase 5 — Evaluation*

Sweep voltage from below the training minimum to well above the training maximum.

- **RF**: flat line at the boundary — physically bounded but wrong
- **MLP (improved)**: continues the curve and stays physically plausible (≥0)

Without the fixes (V², ReLU), the naive MLP went negative at V=34 — physically impossible.
With tanh activation, no polynomial features, and output clipping, the prediction stays reasonable.

In [ ]:
chip_demo   = 5
nozzle_demo = 560
cov_demo    = float(df['Coverage#'].median())
Fr_demo     = float(df['F_r'].median())
dt2_demo    = float(df['dt2'].median())

V_min = float(df['V'].min())
V_max = float(df['V'].max())
V_sweep = np.linspace(V_min - 3, V_max + 7, 120)

def sweep_X_rf(V_arr, Fr, dt2, cov, color_enc, chip, nozzle):
    n = len(V_arr)
    return np.column_stack([
        V_arr, np.full(n, Fr), np.full(n, dt2), np.full(n, cov), np.full(n, color_enc),
        V_arr * Fr, np.full(n, dt2 * cov),
        V_arr ** 2, np.full(n, Fr ** 2),
        np.full(n, chip), np.full(n, nozzle)
    ]).astype(np.float32)

def sweep_X_mlp(V_arr, Fr, dt2, cov, color_enc, chip, nozzle):
    n = len(V_arr)
    return np.column_stack([
        V_arr, np.full(n, Fr), np.full(n, dt2), np.full(n, cov), np.full(n, color_enc),
        V_arr * Fr, np.full(n, dt2 * cov),
        np.full(n, chip), np.full(n, nozzle)
    ]).astype(np.float32)

pred_rf  = rf.predict(sweep_X_rf(V_sweep, Fr_demo, dt2_demo, cov_demo, 1, chip_demo, nozzle_demo))
pred_mlp = np.clip(
    mlp_pipe.predict(sweep_X_mlp(V_sweep, Fr_demo, dt2_demo, cov_demo, 1, chip_demo, nozzle_demo)),
    0, None
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.axvspan(V_min, V_max, alpha=0.1, color='green')
ax.axvline(V_min, color='green', linewidth=1.2, linestyle='--')
ax.axvline(V_max, color='green', linewidth=1.2, linestyle='--',
           label=f'Training range V={V_min:.0f}–{V_max:.0f}')
ax.plot(V_sweep, pred_rf,  color='#00b4d8', linewidth=2, label=f'Random Forest  R²={r2_rf:.4f}')
ax.plot(V_sweep, pred_mlp, color='#e74c3c', linewidth=2, label=f'Neural Network R²={r2_mlp:.4f}')
ax.set_xlabel('Voltage (V)')
ax.set_ylabel('Predicted SD value')
ax.set_title(f'Full V sweep — Chip {chip_demo}, Nozzle {nozzle_demo}, Cyan\nGreen band = training range')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

beyond = V_sweep > V_max
ax2 = axes[1]
ax2.plot(V_sweep[beyond], pred_rf[beyond],  color='#00b4d8', linewidth=2.5,
         label='Random Forest — flat (clips)')
ax2.plot(V_sweep[beyond], pred_mlp[beyond], color='#e74c3c', linewidth=2.5,
         label='Neural Network — smooth extrapolation')
ax2.axvline(V_max, color='green', linewidth=1.2, linestyle='--',
            label=f'Training boundary (V={V_max:.0f})')
ax2.set_xlabel('Voltage (V) — beyond training range')
ax2.set_ylabel('Predicted SD value')
ax2.set_title(f'Zoom: V > {V_max:.0f}\nMLP stays physically plausible (≥0), RF clips flat')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.suptitle('Extrapolation — RF vs improved Neural Network',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Training range: V = {V_min:.0f} to {V_max:.0f}')
for label, preds in [('Random Forest', pred_rf), ('Neural Network', pred_mlp)]:
    at_boundary = preds[np.argmin(np.abs(V_sweep - V_max))]
    at_plus4    = preds[np.argmin(np.abs(V_sweep - (V_max + 4)))]
    print(f'  {label:<20} at V={V_max:.0f}: {at_boundary:.5f}  '
          f'at V={V_max+4:.0f}: {at_plus4:.5f}  change={at_plus4 - at_boundary:+.5f}')

---
## Summary — Key Findings
*CRISP-DM Phase 6 — Deployment*

| Metric | Random Forest (Model 6) | Neural Network (Model 7) |
|---|---|---|
| R² | ~0.9534 | see above |
| MAE | ~0.0148 | see above |
| Extrapolates beyond training range | **No** — clips to boundary | **Yes** — continues curve |
| Requires feature scaling | No | Yes (StandardScaler in Pipeline) |
| Training time | ~2 min | ~10–15 min |

**When to use which model:**
- Within the tested V/F_r/dt2 range → use **Model 6 (RF)**, slightly better R² and faster
- Exploring voltage or flow-rate values outside the tested range → use **Model 7 (MLP)**

**Limitation:** Both models require chip and nozzle identity (HeadIdx, NozzleIdx).
Neither can predict for a completely new printhead batch without retraining.